# Data Preparation: Chat Format (Messages)
This notebook converts the Iris dataset into the JSONL format required for chat-tuning:
```json
{"messages": [
    {"role": "system", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."}
]}
```

In [2]:
import pandas as pd
import json
import os
from sklearn.model_selection import train_test_split
from google.cloud import storage

# --- CONFIGURATION ---
BUCKET_NAME = "mlops-22f3002292" 
DATA_PATH = "../data/iris.csv" 
OUTPUT_DIR = "./data_output"

os.makedirs(OUTPUT_DIR, exist_ok=True)

SYSTEM_PROMPT = "Classify the flower based on its measurements into one of the following species: [Setosa, Versicolor, Virginica]"

/home/jupyter/mlops-22f3002292/.venv/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [3]:
# Load Data
try:
    df = pd.read_csv(DATA_PATH)
    print(f"✅ Loaded iris.csv ({len(df)} rows)")
except FileNotFoundError:
    print("❌ iris.csv not found. Please ensure you have pulled data with DVC.")

✅ Loaded iris.csv (152 rows)


In [4]:
# --- FORMATTING FUNCTION ---
def create_chat_entry(user_content, model_output):
    # Standard OpenAI/Gemini chat format
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": model_output}
        ]
    }

### V1: Raw Numeric Data

In [5]:
def format_v1_raw(row):
    # Matches the user content format from your reference file
    user_text = (
        f"Sepal Length: {row['sepal_length']}, "
        f"Sepal Width: {row['sepal_width']}, "
        f"Petal Length: {row['petal_length']}, "
        f"Petal Width: {row['petal_width']}"
    )
    return create_chat_entry(user_text, row['species'])

v1_dataset = df.apply(format_v1_raw, axis=1).tolist()
print(f"Sample V1:\n{json.dumps(v1_dataset[0], indent=2)}")

Sample V1:
{
  "messages": [
    {
      "role": "system",
      "content": "Classify the flower based on its measurements into one of the following species: [Setosa, Versicolor, Virginica]"
    },
    {
      "role": "user",
      "content": "Sepal Length: 5.1, Sepal Width: 3.5, Petal Length: 1.4, Petal Width: 0.2"
    },
    {
      "role": "assistant",
      "content": "setosa"
    }
  ]
}


### V2: Descriptive Data (Binning)

In [6]:
df_desc = df.copy()
cols = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']

# Convert numbers to Low/Medium/High
for col in cols:
    df_desc[col] = pd.qcut(df_desc[col], q=3, labels=["Low", "Medium", "High"])

def format_v2_desc(row):
    user_text = (
        f"Sepal Length is {row['sepal_length']}, "
        f"Sepal Width is {row['sepal_width']}, "
        f"Petal Length is {row['petal_length']}, "
        f"Petal Width is {row['petal_width']}"
    )
    return create_chat_entry(user_text, row['species'])

v2_dataset = df_desc.apply(format_v2_desc, axis=1).tolist()
print(f"Sample V2:\n{json.dumps(v2_dataset[0], indent=2)}")

Sample V2:
{
  "messages": [
    {
      "role": "system",
      "content": "Classify the flower based on its measurements into one of the following species: [Setosa, Versicolor, Virginica]"
    },
    {
      "role": "user",
      "content": "Sepal Length is Low, Sepal Width is High, Petal Length is Low, Petal Width is Low"
    },
    {
      "role": "assistant",
      "content": "setosa"
    }
  ]
}


In [7]:
# Save and Split
def save_jsonl(data, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, 'w') as f:
        for entry in data:
            json.dump(entry, f)
            f.write('\n')
    return path

train_v1, test_v1 = train_test_split(v1_dataset, test_size=0.2, random_state=42)
path_v1_train = save_jsonl(train_v1, "iris_v1_train.jsonl")
path_v1_test = save_jsonl(test_v1, "iris_v1_test.jsonl")

train_v2, test_v2 = train_test_split(v2_dataset, test_size=0.2, random_state=42)
path_v2_train = save_jsonl(train_v2, "iris_v2_train.jsonl")
path_v2_test = save_jsonl(test_v2, "iris_v2_test.jsonl")

print("✅ Generated JSONL files locally.")

✅ Generated JSONL files locally.


In [8]:
# Upload to GCS
def upload_to_gcs(source_path, dest_blob_name):
    client = storage.Client()
    bucket = client.bucket(BUCKET_NAME)
    blob = bucket.blob(dest_blob_name)
    blob.upload_from_filename(source_path)
    print(f"Uploaded {source_path} to gs://{BUCKET_NAME}/{dest_blob_name}")

print("--- Uploading Datasets ---")
upload_to_gcs(path_v1_train, "week10_finetuning/v1/train.jsonl")
upload_to_gcs(path_v1_test, "week10_finetuning/v1/test.jsonl")
upload_to_gcs(path_v2_train, "week10_finetuning/v2/train.jsonl")
upload_to_gcs(path_v2_test, "week10_finetuning/v2/test.jsonl")

--- Uploading Datasets ---
Uploaded ./data_output/iris_v1_train.jsonl to gs://mlops-22f3002292/week10_finetuning/v1/train.jsonl
Uploaded ./data_output/iris_v1_test.jsonl to gs://mlops-22f3002292/week10_finetuning/v1/test.jsonl
Uploaded ./data_output/iris_v2_train.jsonl to gs://mlops-22f3002292/week10_finetuning/v2/train.jsonl
Uploaded ./data_output/iris_v2_test.jsonl to gs://mlops-22f3002292/week10_finetuning/v2/test.jsonl
